<a href="https://colab.research.google.com/github/Sampath1503/Telecom-Churn-Predictor/blob/main/Churn_XGB_CV_Corrected.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# =============================================================
# CUSTOMER CHURN PREDICTION — LEAKAGE-CONTROLLED XGBOOST
# =============================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTETomek

from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. LOAD DATASET
# ------------------------------------------------------------

df = pd.read_excel("P585 Churn.xlsx")

df.columns = df.columns.str.strip()

df.replace(
    ['Nan', 'nan', 'NaN', 'NAN'],
    np.nan,
    inplace=True
)

df.drop(
    columns=['Unnamed: 0'],
    errors='ignore',
    inplace=True
)


# ------------------------------------------------------------
# 2. FEATURE ENGINEERING
# ------------------------------------------------------------

df['total_charge'] = (
    df['day.charge'] +
    df['eve.charge'] +
    df['night.charge'] +
    df['intl.charge']
)

df['total_mins'] = (
    df['day.mins'] +
    df['eve.mins'] +
    df['night.mins'] +
    df['intl.mins']
)

df['charge_per_min'] = (
    df['total_charge'] / (df['total_mins'] + 1e-6)
)

df['total_calls'] = (
    df['day.calls'] +
    df['eve.calls'] +
    df['night.calls'] +
    df['intl.calls']
)

df['mins_per_call'] = (
    df['total_mins'] / (df['total_calls'] + 1e-6)
)


def get_service_call_bin(calls):
    if pd.isna(calls):
        return np.nan
    elif calls == 0:
        return '0_calls'
    elif calls <= 3:
        return '1-3_calls'
    else:
        return '4+_calls'


df['service_call_bin'] = (
    df['customer.calls'].apply(get_service_call_bin)
)

df['pay_as_you_go_intl'] = (
    (df['intl.plan'] == 'no') &
    (df['intl.mins'] > 0)
).astype(int)


# ------------------------------------------------------------
# 3. FEATURES AND TARGET
# ------------------------------------------------------------

X = df.drop(columns=['churn'])

# Convert target into binary labels
y = (
    df['churn']
    .astype(str)
    .str.strip()
    .str.lower()
    .map({'no': 0, 'yes': 1})
)

if y.isna().any():
    raise ValueError("Unexpected or missing churn target values.")

y = y.astype(int)


# ------------------------------------------------------------
# 4. TRAIN-TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ------------------------------------------------------------
# 5. IDENTIFY FEATURE TYPES
# ------------------------------------------------------------

categorical_cols = [
    'state',
    'voice.plan',
    'intl.plan',
    'service_call_bin'
]

numeric_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)


# ------------------------------------------------------------
# 6. PREPROCESSING PIPELINE
# ------------------------------------------------------------

numeric_pipeline = SklearnPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = SklearnPipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_cols),
    ('categorical', categorical_pipeline, categorical_cols)
])


# ------------------------------------------------------------
# 7. COMPLETE ML PIPELINE
# ------------------------------------------------------------

pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('sampler', SMOTETomek(random_state=42)),
    ('classifier', XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42,
        n_jobs=1
    ))
])


# ------------------------------------------------------------
# 8. HYPERPARAMETER SEARCH
# ------------------------------------------------------------

param_grid = {
    'classifier__n_estimators': [200, 300],
    'classifier__max_depth': [6, 7, 8],
    'classifier__learning_rate': [0.1, 0.05],
    'classifier__subsample': [0.7, 1.0],
    'classifier__colsample_bytree': [0.7, 1.0],
    'classifier__gamma': [0, 0.1, 0.2],
    'classifier__reg_alpha': [0, 0.1],
    'classifier__reg_lambda': [1, 1.5]
}

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_grid,
    n_iter=50,
    scoring='f1',
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1,
    random_state=42,
    refit=True
)


# ------------------------------------------------------------
# 9. MODEL TRAINING
# ------------------------------------------------------------

random_search.fit(X_train, y_train)

print("\nBest Parameters:")
print(random_search.best_params_)

print("\nBest Cross-Validation F1:")
print(random_search.best_score_)

best_model = random_search.best_estimator_


# ------------------------------------------------------------
# 10. FINAL TEST EVALUATION
# ------------------------------------------------------------

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=['no', 'yes']
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

print("\nOverall Metrics:")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))


# ------------------------------------------------------------
# 11. SAVE COMPLETE PIPELINE
# ------------------------------------------------------------

joblib.dump(best_model, 'churn_xgb_pipeline.pkl')

print("\nSaved: churn_xgb_pipeline.pkl")

Numeric features: ['area.code', 'account.length', 'voice.messages', 'intl.mins', 'intl.calls', 'intl.charge', 'day.mins', 'day.calls', 'day.charge', 'eve.mins', 'eve.calls', 'eve.charge', 'night.mins', 'night.calls', 'night.charge', 'customer.calls', 'total_charge', 'total_mins', 'charge_per_min', 'total_calls', 'mins_per_call', 'pay_as_you_go_intl']
Categorical features: ['state', 'voice.plan', 'intl.plan', 'service_call_bin']
Fitting 5 folds for each of 50 candidates, totalling 250 fits


/tmp/ipykernel_842/4078323853.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(



Best Parameters:
{'classifier__subsample': 1.0, 'classifier__reg_lambda': 1.5, 'classifier__reg_alpha': 0.1, 'classifier__n_estimators': 300, 'classifier__max_depth': 6, 'classifier__learning_rate': 0.05, 'classifier__gamma': 0.1, 'classifier__colsample_bytree': 1.0}

Best Cross-Validation F1:
0.9237879495446648

Classification Report:
              precision    recall  f1-score   support

          no       0.98      1.00      0.99       859
         yes       0.97      0.87      0.91       141

    accuracy                           0.98      1000
   macro avg       0.97      0.93      0.95      1000
weighted avg       0.98      0.98      0.98      1000


Confusion Matrix:
[[855   4]
 [ 19 122]]

Overall Metrics:
Accuracy : 0.977
Precision: 0.9682539682539683
Recall   : 0.8652482269503546
F1 Score : 0.9138576779026217
ROC-AUC  : 0.9249250736878606

Saved: churn_xgb_pipeline.pkl
